In [1]:
import sys
sys.path.append("..")
from src import index
from src import image_analysis
import numpy as np
import cv2
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
X1 = index.get_data("../data/processed_images_Tomato/Early_Blight","Early_Blight") + index.get_data("../data/processed_images_Tomato/Healthy","Healthy") + index.get_data("../data/processed_images_Tomato/Late_Blight","Late_Blight") + index.get_data("../data/processed_images_Tomato/Leaf_Mold","Leaf_Mold") + index.get_data("../data/processed_images_Tomato/Septoria_Leaf_Spot","Septoria_Leaf_Spot") + index.get_data("../data/processed_images_Tomato/Spider_Mites","Spider_Mites") + index.get_data("../data/processed_images_Tomato/Target_Spot","Target_Spot") + index.get_data("../data/processed_images_Tomato/Yellow_Leaf_Curl_Virus","Yellow_Leaf_Curl_Virus") + index.get_data("../data/processed_images_Tomato/Mosaic_Virus","Mosaic_Virus")

Y = [x[0] for x in X1]     

In [3]:
features = []
labels   = []
for path, label in X1:
    image = cv2.imread(path)

    if image is None:   
        raise ValueError(f"Image at path {path} could not be loaded.")
    
    hist = image_analysis.get_histogram(image)
    glcm = image_analysis.GLCM(image) 
    combined = np.concatenate([hist, glcm])
    features.append(combined)
    labels.append(label)

X = np.array(features)   # shape (n_images, n_features)
y = np.array(labels)     # shape (n_images,)

print(X.shape, y.shape)

(12817, 184) (12817,)


In [4]:
liste = [10, 23, 35, 42, 59]

for i in range(5):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=liste[i])

    scaler = StandardScaler()

    scaler.fit(X_train)
    X_train = scaler.transform(X_train)
    X_test  = scaler.transform(X_test)

    clf = SVC(kernel="rbf")

    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    print(classification_report(y_test, y_pred))
    print(confusion_matrix(y_test, y_pred))

                        precision    recall  f1-score   support

          Early_Blight       0.84      0.78      0.81       200
               Healthy       0.95      0.96      0.96       318
           Late_Blight       0.86      0.88      0.87       382
             Leaf_Mold       0.88      0.88      0.88       191
          Mosaic_Virus       0.93      0.87      0.90        75
    Septoria_Leaf_Spot       0.85      0.81      0.83       354
          Spider_Mites       0.83      0.91      0.87       335
           Target_Spot       0.84      0.86      0.85       281
Yellow_Leaf_Curl_Virus       0.94      0.93      0.93       428

              accuracy                           0.88      2564
             macro avg       0.88      0.87      0.88      2564
          weighted avg       0.88      0.88      0.88      2564

[[155   0  25   1   0   6   3   7   3]
 [  0 305   4   0   0   4   1   4   0]
 [ 12   3 335   7   0   8   9   0   8]
 [  1   0   7 169   0   7   1   1   5]
 [  0   0

In [5]:
liste = [10, 23, 35, 42, 59]

for i in range(5):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=liste[i])

    rdm_f = RandomForestClassifier(n_estimators=100, random_state=42)

    rdm_f.fit(X_train, y_train)

    y_pred = rdm_f.predict(X_test)
    print(classification_report(y_test, y_pred))
    print(confusion_matrix(y_test, y_pred))

                        precision    recall  f1-score   support

          Early_Blight       0.87      0.80      0.84       200
               Healthy       0.96      0.98      0.97       318
           Late_Blight       0.89      0.88      0.88       382
             Leaf_Mold       0.92      0.91      0.91       191
          Mosaic_Virus       0.93      0.89      0.91        75
    Septoria_Leaf_Spot       0.88      0.86      0.87       354
          Spider_Mites       0.87      0.93      0.90       335
           Target_Spot       0.87      0.91      0.89       281
Yellow_Leaf_Curl_Virus       0.95      0.92      0.93       428

              accuracy                           0.90      2564
             macro avg       0.90      0.90      0.90      2564
          weighted avg       0.90      0.90      0.90      2564

[[160   0  21   0   0   5   1   8   5]
 [  0 312   2   0   0   0   1   3   0]
 [ 10   3 335   5   0  12   9   0   8]
 [  1   0   8 174   0   4   0   2   2]
 [  1   0

In [7]:
feature_importances = rdm_f.feature_importances_

print("texture (last 5):", feature_importances[-5:].sum())
print("color (first 179):", feature_importances[:179].sum())

texture (last 5): 0.10270958257225389
color (first 179): 0.8972904174277461
